In [1]:
!pip install -q tensorflow pillow pandas scikit-learn seaborn imbalanced-learn

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
import os
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
from sklearn.model_selection import KFold
from sklearn.metrics import (confusion_matrix, accuracy_score,
                             precision_score, recall_score,
                             f1_score, classification_report,
                             roc_curve, auc)
from imblearn.metrics import specificity_score
import tensorflow as tf
from tensorflow.keras import mixed_precision
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.optimizers import Adam

# Mixed Precision für schnelleres Training
policy = mixed_precision.Policy('mixed_float16')
mixed_precision.set_global_policy(policy)
print("TensorFlow:", tf.__version__)

TensorFlow: 2.19.0


In [4]:
import os

# Zeigt alles was in drive verfügbar ist
print("── /content/drive ──")
for item in os.listdir('/content/drive'):
    print(f"  {item}")

print("\n── /content/drive/MyDrive ──")
for item in os.listdir('/content/drive/MyDrive'):
    print(f"  {item}")

── /content/drive ──
  Othercomputers
  MyDrive
  .shortcut-targets-by-id
  .Trash-0
  .Encrypted

── /content/drive/MyDrive ──
  Colab Notebooks
  average_raster_output_with_grid_views_500m.tif
  count_faves_map.ipynb
  barplot_photo_count_L1:L2.R
  Map_Count_Views.ipynb
  Map_Count_Faves.ipynb
  photo_numbers_10m.ipynb
  flickr_data_subset (1)
  SM_Maps_Dez_2024
  NDVI.R
  EO_Browser_images
  EO_Browser_images-2
  EO_Browser_images-3
  EO_Browser_images-8.zip
  EO_Browser_images-9.zip
  EO_Browser_images-7.zip
  A49_Trasse.gpkg
  Dannenroederforst.gpkg
  Species_common_names_new
  Netra_Maps.png
  3_maps_map.png
  Paper_Netra_Map_Templates.qgz
  Moshi_Siha_partic_proj.qgz
  Siha_Moshi_Maps_Feb
  Moshi_Siha_partic_proj_Feb.qgz
  Shp_kili_proj
  Zero_Shot
  Zero_Shot_Results.csv
  Zero_Shot_Scores.csv
  Examples
  Zero Shot
  indicators_and_drivers_January25_2.xlsx
  Moshi_Siha_Maps_Feb_17
  Moshi_Siha_Maps_Feb_24
  Moshi_Siha_Maps_March_2
  Organischer Kohlenstoff im Einzugsgebiet
  f

In [6]:
# Pfade anpassen!
base_dir    = '/content/drive/MyDrive/flickr_data_subset (1)/subset/'

# Bildordner (mit augmentierten Bildern)
indoor_dir  = os.path.join(base_dir, 'indoor/indoor')
outdoor_dir = os.path.join(base_dir, 'outdoor/outdoor')

# Ausgabeordner
models_dir  = os.path.join(base_dir, 'resnet_models')
plots_dir   = os.path.join(models_dir, 'plots')
metrics_dir = os.path.join(models_dir, 'metrics')

os.makedirs(models_dir,  exist_ok=True)
os.makedirs(plots_dir,   exist_ok=True)
os.makedirs(metrics_dir, exist_ok=True)

# Bildanzahl prüfen
VALID_EXT = ('.jpg', '.jpeg', '.png', '.bmp', '.webp')
n_indoor  = sum(1 for f in os.listdir(indoor_dir)  if f.lower().endswith(VALID_EXT))
n_outdoor = sum(1 for f in os.listdir(outdoor_dir) if f.lower().endswith(VALID_EXT))

print(f"Indoor:  {n_indoor} Bilder")
print(f"Outdoor: {n_outdoor} Bilder")
print(f"Gesamt:  {n_indoor + n_outdoor} Bilder")

Indoor:  2500 Bilder
Outdoor: 2500 Bilder
Gesamt:  5000 Bilder


In [7]:
import tensorflow as tf

gpus = tf.config.list_physical_devices('GPU')
print("GPUs:", gpus)

# Sollte dann ausgeben:
# GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]

GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [8]:
label_map = {'indoor': 0, 'outdoor': 1}
folders   = {'indoor': indoor_dir, 'outdoor': outdoor_dir}

X = []
y = []

for label, folder in folders.items():
    print(f"Lade {label}...")
    for fname in sorted(os.listdir(folder)):
        if not fname.lower().endswith(VALID_EXT):
            continue
        img_path = os.path.join(folder, fname)
        img = cv2.imread(img_path)
        if img is None:
            continue
        img = cv2.resize(img, (224, 224))
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        X.append(img)
        y.append(label_map[label])

X = np.array(X, dtype='float32') / 255.0
y = np.array(y)

print(f"\nGeladen: {len(X)} Bilder")
print(f"Indoor:  {np.sum(y == 0)}")
print(f"Outdoor: {np.sum(y == 1)}")

y_cat = to_categorical(y, num_classes=2)

Lade indoor...
Lade outdoor...

Geladen: 5000 Bilder
Indoor:  2500
Outdoor: 2500


In [ ]:
# Parameter
BATCH_SIZE = 32
EPOCHS     = 30
SEED       = 7
today      = datetime.today().strftime('%Y-%m-%d')

np.random.seed(SEED)
kfold = KFold(n_splits=5, shuffle=True, random_state=SEED)

# Augmentation (nur auf Trainingsdaten)
datagen = ImageDataGenerator(
    rotation_range=20,
    horizontal_flip=True,
    width_shift_range=0.2,
    height_shift_range=0.2,
    zoom_range=0.2
)

# Metriken sammeln
all_accuracies   = []
all_f1           = []
all_sensitivity  = []
all_specificity  = []
all_cm           = []

for fold, (train_idx, test_idx) in enumerate(kfold.split(X, y_cat), 1):

    # ── Bereits trainierte Folds überspringen ──────────────
    checkpoint_path = os.path.join(models_dir, f"resnet50_{today}_fold{fold}.keras")
    if os.path.exists(checkpoint_path):
        print(f"✓ Fold {fold} bereits vorhanden → übersprungen")
        metrics_path = os.path.join(metrics_dir, f'metrics_fold{fold}_{today}.csv')
        if os.path.exists(metrics_path):
            m = pd.read_csv(metrics_path)
            all_accuracies.append(m['Accuracy'].values[0])
            all_f1.append(m['F1'].values[0])
            all_sensitivity.append(m['Recall'].values[0])
            all_specificity.append(m['Specificity'].values[0])
            all_cm.append(None)
        continue
    # ───────────────────────────────────────────────────────

    print(f"\n── Fold {fold} ────────────────────────────")

    # Modell definieren
    base_model = ResNet50(
        include_top=False, pooling="max",
        input_shape=(224, 224, 3), weights="imagenet"
    )
    for layer in base_model.layers:
        layer.trainable = False

    model = Sequential([
        base_model,
        Dense(128, activation='relu'),
        Dense(2, activation='softmax')
    ])

    model.compile(
        optimizer=Adam(learning_rate=1e-4),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )

    # Callbacks
    es = EarlyStopping(monitor='val_loss', patience=10, verbose=1, restore_best_weights=True)
    cp = ModelCheckpoint(checkpoint_path, monitor='val_loss', save_best_only=True, verbose=1)

    # Training
    history = model.fit(
        datagen.flow(X[train_idx], y_cat[train_idx], batch_size=BATCH_SIZE),
        validation_data=(X[test_idx], y_cat[test_idx]),
        epochs=EPOCHS,
        callbacks=[es, cp],
        verbose=1
    )

    # Bestes Modell laden
    best_model = tf.keras.models.load_model(checkpoint_path)

    # Evaluation
    y_pred_probs = best_model.predict(X[test_idx], verbose=0)
    y_pred       = np.argmax(y_pred_probs, axis=1)
    y_true       = np.argmax(y_cat[test_idx], axis=1)

    acc  = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred)
    rec  = recall_score(y_true, y_pred)
    f1   = f1_score(y_true, y_pred)
    spec = specificity_score(y_true, y_pred)
    cm   = confusion_matrix(y_true, y_pred)

    print(f"Accuracy:    {acc:.4f}")
    print(f"F1-Score:    {f1:.4f}")
    print(f"Sensitivity: {rec:.4f}")
    print(f"Specificity: {spec:.4f}")

    all_accuracies.append(acc)
    all_f1.append(f1)
    all_sensitivity.append(rec)
    all_specificity.append(spec)
    all_cm.append(cm)

    # Confusion Matrix Plot speichern
    plt.figure(figsize=(5, 4))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=['indoor', 'outdoor'],
                yticklabels=['indoor', 'outdoor'])
    plt.xlabel('Predicted')
    plt.ylabel('True')
    plt.title(f'Confusion Matrix – Fold {fold}')
    plt.tight_layout()
    plt.savefig(os.path.join(plots_dir, f'cm_fold{fold}_{today}.png'))
    plt.close()

    # Accuracy/Loss Plot
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))
    ax1.plot(history.history['accuracy'],     label='Train')
    ax1.plot(history.history['val_accuracy'], label='Val')
    ax1.set_title(f'Accuracy – Fold {fold}')
    ax1.legend()
    ax2.plot(history.history['loss'],     label='Train')
    ax2.plot(history.history['val_loss'], label='Val')
    ax2.set_title(f'Loss – Fold {fold}')
    ax2.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(plots_dir, f'loss_fold{fold}_{today}.png'))
    plt.close()

    # Metriken als CSV
    pd.DataFrame([{
        'Fold': fold, 'Accuracy': acc, 'Precision': prec,
        'Recall': rec, 'Specificity': spec, 'F1': f1
    }]).to_csv(os.path.join(metrics_dir, f'metrics_fold{fold}_{today}.csv'), index=False)

print("\n── Training abgeschlossen ──────────────────")

✓ Fold 1 bereits vorhanden → übersprungen
✓ Fold 2 bereits vorhanden → übersprungen
✓ Fold 3 bereits vorhanden → übersprungen

── Fold 4 ────────────────────────────
94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step
Epoch 1/30
125/125 ━━━━━━━━━━━━━━━━━━━━ 0s 365ms/step - accuracy: 0.5690 - loss: 0.6871
Epoch 1: val_loss improved from None to 0.57824, saving model to /content/drive/MyDrive/flickr_data_subset (1)/subset/resnet_models/resnet50_2026-04-20_fold4.keras

Epoch 1: finished saving model to /content/drive/MyDrive/flickr_data_subset (1)/subset/resnet_models/resnet50_2026-04-20_fold4.keras
125/125 ━━━━━━━━━━━━━━━━━━━━ 79s 476ms/step - accuracy: 0.6155 - loss: 0.6579 - val_accuracy: 0.7480 - val_loss: 0.5782
Epoch 2/30
125/125 ━━━━━━━━━━━━━━━━━━━━ 0s 381ms/step - accuracy: 0.6874 - loss: 0.6026
Epoch 2: val_loss improved from 0.57824 to 0.54259, saving model to /content/drive/MyDrive/flickr_data_subset (1)/subset/resnet_models/resnet50_2026-04-20_fold4.keras

Epoch 2: finished sav

In [ ]:
print("── Gesamtergebnisse ────────────────────────")
print(f"Ø Accuracy:    {np.mean(all_accuracies):.4f} ± {np.std(all_accuracies):.4f}")
print(f"Ø F1-Score:    {np.mean(all_f1):.4f} ± {np.std(all_f1):.4f}")
print(f"Ø Sensitivity: {np.mean(all_sensitivity):.4f} ± {np.std(all_sensitivity):.4f}")
print(f"Ø Specificity: {np.mean(all_specificity):.4f} ± {np.std(all_specificity):.4f}")

# Zusammenfassung als CSV
pd.DataFrame({
    'Fold':        range(1, 6),
    'Accuracy':    all_accuracies,
    'F1':          all_f1,
    'Sensitivity': all_sensitivity,
    'Specificity': all_specificity
}).to_csv(os.path.join(metrics_dir, f'summary_{today}.csv'), index=False)

print(f"\nAlles gespeichert in: {models_dir}")

In [ ]:
# Bestes Modell laden (Fold mit höchstem F1)
best_fold = np.argmax(all_f1) + 1
best_model_path = os.path.join(models_dir, f"resnet50_{today}_fold{best_fold}.keras")
best_model = tf.keras.models.load_model(best_model_path)
print(f"Bestes Modell: Fold {best_fold} (F1 = {all_f1[best_fold-1]:.4f})")

# Neue Bilder klassifizieren
new_image_folder = "/content/drive/MyDrive/flickr_data_subset/alle_bilder"
categories = ['indoor', 'outdoor']
results = []

for fname in sorted(os.listdir(new_image_folder)):
    if not fname.lower().endswith(VALID_EXT):
        continue
    img = cv2.imread(os.path.join(new_image_folder, fname))
    if img is None:
        continue
    img = cv2.resize(img, (224, 224))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = img.astype('float32') / 255.0
    img = np.expand_dims(img, axis=0)

    pred_probs = best_model.predict(img, verbose=0)
    pred_label = categories[np.argmax(pred_probs)]
    confidence = np.max(pred_probs)

    results.append({'file_name': fname, 'pred_label': pred_label, 'confidence': confidence})

df_results = pd.DataFrame(results)
df_results.to_csv(os.path.join(base_dir, 'resnet_results.csv'), index=False)

print("Verteilung:")
print(df_results['pred_label'].value_counts())
print(f"\nGesamt: {len(df_results)} Bilder")